In [ ]:
"""
Workstation rental price tier  -  simple, CPU-only solution
Libraries: pandas, numpy, scikit-learn only.   Target runtime: < 3 minutes.

Idea in 5 steps
  1. Keep ALL rows (no dropna) - HistGradientBoosting handles missing values itself.
  2. Turn raw columns into better features: dates, the facilities list, ratios,
     "this space compared with its zone".
  3. Add "what do similar spaces cost?" features learned from the labels:
       - nearest-neighbour average tier (by geo_lat / geo_lon)
       - average tier of the same operator / zone / zone+class
     These are built "out of fold" so a row never sees its own label.
  4. Train a gradient boosting classifier.
  5. Tune 4 small class multipliers to maximise weighted F1, write submission.csv
"""
import time
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import f1_score

# ------------------------------------------------------------------ settings
DIR = "/kaggle/input/competitions/workstation-rental-price-tier-prediction/kaggle_ready_dataset/"
OUT = "submission.csv"
VALIDATE = True            # True: also print an honest validation score (+~40 s)
N_FOLDS = 5                # folds used to build the label-based features
K_NEIGHBORS = 20           # neighbours used for the geo feature
SMOOTH = 10                # shrinks group averages that rest on few rows
UNSEEN_OPERATOR_SHARE = 0.35   # test has ~34% operators never seen in train
MAX_ITER = 200             # boosting rounds (early stopping may stop sooner)
SEED = 42

t0 = time.time()


def tick(msg):
    print(f"[{time.time() - t0:6.1f}s] {msg}", flush=True)


# ------------------------------------------------------------------ 1. load
train = pd.read_csv(DIR + "train.csv")
test = pd.read_csv(DIR + "test.csv")
n_train = len(train)
y = train["price_tier"].values
test_ids = test["space_id"].values

# stack train+test so counts / encodings are built the same way for both
data = pd.concat([train.drop(columns=["price_tier"]), test], ignore_index=True)
data = data.drop(columns=["space_id"])
data["facilities"] = data["facilities"].fillna("")
tick(f"loaded train {train.shape}, test {test.shape}")

# ------------------------------------------------------------------ 2. features
X = data.select_dtypes("number").copy()          # all numeric columns as they are

# t / f columns -> 1 / 0 (missing stays missing)
for c in ["operator_certified", "operator_has_avatar", "operator_id_verified", "bookable"]:
    X[c] = data[c].map({"t": 1.0, "f": 0.0})

# text categories -> pandas 'category' (the model treats them as categories)
for c in ["operator_class", "access_type", "compliance_status", "metro", "space_class", "borough"]:
    X[c] = data[c].astype("category")
X["zone_num"] = data["zone"].str[3:].astype(float)      # 'ZN-0557' -> 557

# dates -> numbers
ref = pd.Timestamp("2024-01-01")
d_start = pd.to_datetime(data["snapshot_start_date"], errors="coerce")
d_end = pd.to_datetime(data["snapshot_end_date"], errors="coerce")
d_first = pd.to_datetime(data["first_review_date"], errors="coerce")
d_last = pd.to_datetime(data["last_review_date"], errors="coerce")
X["snap_start"] = (d_start - ref).dt.days
X["snap_len"] = (d_end - d_start).dt.days
X["snap_month"] = d_start.dt.month
X["snap_weekday"] = d_start.dt.dayofweek
X["days_since_first_review"] = (d_start - d_first).dt.days
X["days_since_last_review"] = (d_start - d_last).dt.days
X["review_span"] = (d_last - d_first).dt.days

# a few ratios the trees cannot easily build themselves
X["avail_30"] = data["open_slots_a"] / 30
X["avail_365"] = data["open_slots_d"] / 365
X["review_rate_12m"] = data["review_count_12m"] / (data["review_count"] + 1)
X["seats_per_workstation"] = data["max_seats"] / data["workstations"]

# how common is this operator / zone, and how does this space compare with its zone?
X["n_operator"] = data.groupby("operator_ref")["zone"].transform("size")
X["n_zone"] = data.groupby("zone")["metro"].transform("size")
for c in ["max_seats", "workstations", "zones", "utility_rooms"]:
    X[c + "_vs_zone"] = data[c] - data.groupby("zone")[c].transform("median")

# facilities: '["FCL-087", "FCL-007", ...]'  ->  one 0/1 column per common facility
X["n_facilities"] = data["facilities"].str.count("FCL-")
cv = CountVectorizer(token_pattern=r"FCL-\d+", lowercase=False, binary=True, max_features=60)
F = cv.fit_transform(data["facilities"]).astype(np.uint8).toarray()
fac = pd.DataFrame(F, columns=["fac_" + n for n in cv.get_feature_names_out()], index=data.index)
X = pd.concat([X, fac], axis=1)
tick(f"base features done: {X.shape[1]} columns")

# ------------------------------------------------------------------ 3. label-based features
# keys we average the tier over
keys = {
    "operator": data["operator_ref"].fillna("missing"),
    "zone": data["zone"],
    "zone_class": data["zone"] + "|" + data["space_class"],
    "zone_seats": data["zone"] + "|" + data["max_seats"].astype(str),
}
xy = data[["geo_lat", "geo_lon"]].values


def label_features(fit_idx, apply_idx):
    """Learn from the labelled rows `fit_idx`, create features for rows `apply_idx`."""
    y_fit = y[fit_idx]
    prior = y_fit.mean()
    out = {}
    # (a) smoothed average tier of the same operator / zone / ...
    for name, key in keys.items():
        stats = pd.DataFrame({"k": key.values[fit_idx], "y": y_fit}).groupby("k")["y"].agg(["sum", "count"])
        k = pd.Series(key.values[apply_idx])
        s = k.map(stats["sum"]).fillna(0).values
        c = k.map(stats["count"]).fillna(0).values
        out["te_" + name] = (s + SMOOTH * prior) / (c + SMOOTH)
    # (b) average tier of the closest labelled spaces on the map
    nn = NearestNeighbors(n_neighbors=K_NEIGHBORS, n_jobs=-1).fit(xy[fit_idx])
    _, nb = nn.kneighbors(xy[apply_idx])
    near = y_fit[nb]
    out["knn_5"] = near[:, :5].mean(axis=1)
    out["knn_20"] = near.mean(axis=1)
    return pd.DataFrame(out, index=apply_idx)


# Out-of-fold: each train row gets features computed WITHOUT its own fold's labels.
# Test has ~34% operators never seen in train, so for ~35% of operators we keep all
# their rows in the same fold -> train rows face the same situation as test rows.
rng = np.random.RandomState(SEED)
op_code = pd.factorize(keys["operator"].values[:n_train])[0]
keep_together = rng.rand(op_code.max() + 1) < UNSEEN_OPERATOR_SHARE
op_fold = rng.randint(0, N_FOLDS, op_code.max() + 1)
row_fold = rng.randint(0, N_FOLDS, n_train)
fold = np.where(keep_together[op_code], op_fold[op_code], row_fold)

parts = [label_features(np.where(fold != f)[0], np.where(fold == f)[0]) for f in range(N_FOLDS)]
parts.append(label_features(np.arange(n_train), np.arange(n_train, len(data))))   # test rows
X = pd.concat([X, pd.concat(parts).sort_index()], axis=1)
X = X.replace([np.inf, -np.inf], np.nan)
tick(f"label-based features done: {X.shape[1]} columns in total")

X_train, X_test = X.iloc[:n_train], X.iloc[n_train:]


# ------------------------------------------------------------------ 4. model
def make_model():
    return HistGradientBoostingClassifier(
        max_iter=MAX_ITER, learning_rate=0.15, max_leaf_nodes=48, min_samples_leaf=50,
        l2_regularization=1.0, early_stopping=True, validation_fraction=0.1,
        n_iter_no_change=20, categorical_features="from_dtype", random_state=SEED)


def tune_weights(P, y_true, classes):
    """Multiply class probabilities by small weights to maximise weighted F1."""
    w = np.ones(P.shape[1])
    best = f1_score(y_true, classes[(P * w).argmax(1)], average="weighted")
    for _ in range(2):
        for k in range(P.shape[1]):
            for v in np.arange(0.7, 1.5, 0.1):
                w2 = w.copy()
                w2[k] = v
                s = f1_score(y_true, classes[(P * w2).argmax(1)], average="weighted")
                if s > best:
                    best, w = s, w2
    return w, best


weights = np.ones(4)
models = []

if VALIDATE:
    # hold out fold 0: its features were built exactly like the test features
    val = fold == 0
    m1 = make_model().fit(X_train[~val], y[~val])
    P_val = m1.predict_proba(X_train[val])
    classes = m1.classes_
    base = f1_score(y[val], classes[P_val.argmax(1)], average="weighted")
    weights, tuned = tune_weights(P_val, y[val], classes)
    tick(f"VALIDATION weighted F1: {base:.4f}  | after class-weight tuning: {tuned:.4f}")
    print("   class weights:", np.round(weights, 2))
    models.append(m1)

m2 = make_model().fit(X_train, y)
models.append(m2)
classes = m2.classes_
tick("final model trained")

# ------------------------------------------------------------------ 5. submission
P_test = np.mean([m.predict_proba(X_test) for m in models], axis=0)   # average of the models
pred = classes[(P_test * weights).argmax(1)]

sub = pd.read_csv(DIR + "sample_submission.csv")
id_col, target_col = sub.columns[0], sub.columns[1]
sub[target_col] = pd.Series(pred, index=test_ids).reindex(sub[id_col].values).values
sub.to_csv(OUT, index=False)
tick(f"saved {OUT}  |  predicted tier shares: {dict(sub[target_col].value_counts(normalize=True).round(3).sort_index())}")